# 02. Data Preprocessing & Feature Engineering

## Mục tiêu:
Notebook này trực quan hóa các bước tiền xử lý dữ liệu và tạo đặc trưng đã được đóng gói trong các tệp mã nguồn chuyên dụng `src/data/cleaner.py` và `src/features/builder.py`.

Các bước thực hiện bao gồm:
1. Load cấu hình và nạp dữ liệu thô.
2. Xử lý các giá trị khuyết thiếu (Missing Values).
3. Thiết lập các đặc trưng dựa trên thời gian (Time-based features).
4. Khám phá các Điểm kỳ dị (Outliers) bằng Z-score.
5. Xây dựng các cấu trúc dữ liệu cần thiết cho Khai phá, ví dụ: Phân rổ dữ liệu (Basket Data) và Daily Profiles. 

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào dường dẫn hệ thống để gọi file trong src
sys.path.append(os.path.abspath('..'))

from src.data.loader import load_config, load_raw_data
from src.data.cleaner import handle_missing_values, detect_outliers, add_time_features
from src.features.builder import discretize_continuous, create_basket_data, create_daily_profiles

import warnings
warnings.filterwarnings('ignore')

## 1. Nạp dữ liệu thô (Data Loading)

In [2]:
# Load config
config = load_config('../configs/params.yaml')

# Load raw data
DATA_PATH = '../household_power_consumption.txt'
df_raw = load_raw_data(DATA_PATH, config)

print("Dữ liệu ban đầu (Raw Data):")
display(df_raw.head())

Loaded 2075259 records from ../household_power_consumption.txt
Date range: 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Missing values:
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
Dữ liệu ban đầu (Raw Data):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 2. Tiền xử lý Dữ liệu (Preprocessing)
Sử dụng các phương pháp tuyến tính (linear) để lấp sự thiếu hụt. Phương thức lấp đầy đã khai báo trong cấu hình params.yaml.

In [3]:
# Xử lý missing values
interp_method = config['preprocessing']['interpolation_method']
df_clean = handle_missing_values(df_raw, method=interp_method)

print("\nMẫu dữ liệu sau lấp Missing Values:")
display(df_clean.head())

Missing values after cleaning: 0

Mẫu dữ liệu sau lấp Missing Values:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 3. Tạo đặc trưng Thời gian (Time-based Features)

In [4]:
# Trích xuất hour, day, is_weekend, peak_hours...
df_enhanced = add_time_features(df_clean)

print("Các cột hiện có:")
print(df_enhanced.columns.tolist())

display(df_enhanced[['hour', 'day_of_week', 'is_weekend', 'is_peak_hour']].head())

Các cột hiện có:
['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'hour', 'day_of_week', 'day_of_month', 'month', 'year', 'is_weekend', 'is_peak_hour']


,hour,day_of_week,is_weekend,is_peak_hour
datetime,,,,
2006-12-16 17:24:00,17,5,1,0
2006-12-16 17:25:00,17,5,1,0
2006-12-16 17:26:00,17,5,1,0
2006-12-16 17:27:00,17,5,1,0
2006-12-16 17:28:00,17,5,1,0


## 4. Phát hiện Ngoại lai bằng Phân bố Hình học (Z-Score Outliers)

In [5]:
# Tìm các điểm tiêu thụ bất thường (vượt quá 3 Std)
outliers = detect_outliers(df_enhanced, ['Global_active_power', 'Global_intensity'], threshold=3.0)
df_outliers = df_enhanced[outliers['Global_active_power']]

print(f"Số lượng điểm vượt chuẩn quá 3 lần Std: {len(df_outliers)}")
display(df_outliers.head())

Số lượng điểm vượt chuẩn quá 3 lần Std: 36671


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,day_of_week,day_of_month,month,year,is_weekend,is_peak_hour
datetime,,,,,,,,,,,,,,
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,5,16,12,2006,1,0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,5,16,12,2006,1,0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,5,16,12,2006,1,0
2006-12-16 17:34:00,4.448,0.498,232.86,19.6,0.0,1.0,17.0,17,5,16,12,2006,1,0
2006-12-16 17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,17,5,16,12,2006,1,0


## 5. Dữ liệu Phân rổ cho Luật Kết hợp (Association Rules Basket)
Để áp dụng thuật toán Apriori, chúng ta rời rạc hoá giá trị số thành giá trị phân loại ('Low', 'Medium', 'High').

In [6]:
# Khởi tạo Basket Data (Chuyển thành biến Binary với Get_Dummies)
basket_encoded = create_basket_data(df_enhanced, config)

print(f"Kích thước tệp Basket Transaction: {basket_encoded.shape}")
display(basket_encoded.head())

Kích thước tệp Basket Transaction: (2075259, 21)


,Power_Level_Low,Power_Level_Medium,Power_Level_High,Intensity_Level_Low,Intensity_Level_Medium,Intensity_Level_High,SubMeter1_Level_Low,SubMeter1_Level_Medium,SubMeter1_Level_High,SubMeter2_Level_Low,...,SubMeter2_Level_High,SubMeter3_Level_Low,SubMeter3_Level_Medium,SubMeter3_Level_High,Hour_Period_Night,Hour_Period_Morning,Hour_Period_Afternoon,Hour_Period_Evening,Day_Type_Weekday,Day_Type_Weekend
datetime,,,,,,,,,,,,,,,,,,,,,
2006-12-16 17:24:00,False,True,False,False,True,False,True,False,False,True,...,False,False,True,False,False,False,True,False,False,True
2006-12-16 17:25:00,False,True,False,False,True,False,True,False,False,True,...,False,False,True,False,False,False,True,False,False,True
2006-12-16 17:26:00,False,True,False,False,True,False,True,False,False,True,...,False,False,True,False,False,False,True,False,False,True
2006-12-16 17:27:00,False,True,False,False,True,False,True,False,False,True,...,False,False,True,False,False,False,True,False,False,True
2006-12-16 17:28:00,True,False,False,True,False,False,True,False,False,True,...,False,False,True,False,False,False,True,False,False,True


## 6. Daily Profiles cho K-Means (Clustering Profiles)
Biến đổi chuỗi thời gian phút thành tổng hợp từng ngày (24 giờ).

In [7]:
# Xây dựng ma trận cho K-Means (Mỗi phần tử là 1 ngày, mỗi chiều đại diện giờ thứ N trong ngày)
daily_profiles = create_daily_profiles(df_enhanced)

print(f"\nĐã tạo thành công Profile tiêu thụ cho {len(daily_profiles)} ngày.")
display(daily_profiles.head())

Created 1440 daily profiles

Đã tạo thành công Profile tiêu thụ cho 1440 ngày.


,day_of_week,is_weekend,total_consumption,mean_consumption,std_consumption,max_consumption,min_consumption,hour_0,hour_1,hour_2,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
date,,,,,,,,,,,,,,,,,,,,,
2006-12-17,6,1,56.507667,2.354486,0.862821,3.697100,0.437733,1.882467,3.349400,1.587267,...,2.092633,2.985400,3.326033,3.406767,3.697100,2.908400,3.361500,3.040767,1.518000,0.437733
2006-12-18,0,0,36.730433,1.530435,0.817779,3.050567,0.276367,0.276367,0.313300,0.284467,...,1.733033,1.784300,1.949300,2.154900,2.402533,2.614500,3.050567,2.169733,1.738800,1.547267
2006-12-19,1,0,27.769900,1.157079,0.993433,3.879033,0.300467,0.837133,0.353033,0.327233,...,0.302133,0.421367,1.372133,2.111500,2.204700,1.842100,2.940533,1.442867,0.720000,0.383700
2006-12-20,2,0,37.095800,1.545658,1.204481,3.646067,0.258667,0.459833,0.258667,0.784367,...,1.294900,0.281133,0.468433,0.573500,2.836833,3.248633,3.575467,3.646067,3.058967,2.381767
2006-12-21,3,0,28.618567,1.192440,0.771635,2.575800,0.246733,1.535867,1.397967,1.274900,...,1.023900,0.307400,1.360067,1.752633,2.443300,2.197133,2.437367,0.982267,0.280267,0.270433


## Nhận xét
Việc tách biệt code thành các tệp `.py` gọn gàng hỗ trợ module hoá cao, đặc biệt cho các file phân tích sau. Nhờ vậy, ở notebook `03_mining` hay `04_modeling`, chúng ta không cần viêt lại toàn bộ các hàm này. Dữ liệu có thể được xử lý trực tiếp một cách nhanh chóng.